# WHOOP Data Explorer & Structure Analysis
This notebook explores the raw WHOOP JSON data to understand its structure and determine the best storage strategy.

In [1]:
import json
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# Load the data
data_path = Path('../data/raw/whoop_complete_20260204_232639.json')
with open(data_path, 'r') as f:
    raw_data = json.load(f)

print(f"Top-level keys: {list(raw_data.keys())}")

Top-level keys: ['user', 'body', 'cycles', 'recovery', 'sleep', 'workouts']


## 1. Data Structure Overview

In [2]:
# Count records in each category
print("=" * 50)
print("DATA SUMMARY")
print("=" * 50)
print(f"\nUser Info: {raw_data.get('user', {})}")
print(f"\nBody Info: {raw_data.get('body', {})}")
print(f"\nCycles:    {len(raw_data.get('cycles', []))} records")
print(f"Recovery:  {len(raw_data.get('recovery', []))} records")
print(f"Sleep:     {len(raw_data.get('sleep', []))} records")
print(f"Workouts:  {len(raw_data.get('workouts', []))} records")

DATA SUMMARY

User Info: {'user_id': 13410036, 'email': 'levijbdavis@gmail.com', 'first_name': 'Levi', 'last_name': 'Davis'}

Body Info: {'height_meter': 1.7272, 'weight_kilogram': 68.0388, 'max_heart_rate': 191}

Cycles:    181 records
Recovery:  179 records
Sleep:     195 records
Workouts:  74 records


## 2. Explore Each Data Type

### 2.1 Cycles (Daily Strain)

In [3]:
# Examine a single cycle record
sample_cycle = raw_data['cycles'][0]
print("CYCLE STRUCTURE:")
print(json.dumps(sample_cycle, indent=2))

CYCLE STRUCTURE:
{
  "id": 1292957107,
  "user_id": 13410036,
  "created_at": "2026-02-04T19:38:34.292Z",
  "updated_at": "2026-02-04T19:43:05.365Z",
  "start": "2026-02-04T07:31:08.530Z",
  "end": null,
  "timezone_offset": "-05:00",
  "score_state": "SCORED",
  "score": {
    "strain": 4.426687,
    "kilojoule": 6169.0913,
    "average_heart_rate": 57,
    "max_heart_rate": 128
  }
}


In [4]:
# Flatten cycles to DataFrame
def flatten_cycles(cycles):
    records = []
    for c in cycles:
        record = {
            'cycle_id': c['id'],
            'start': c['start'],
            'end': c['end'],
            'timezone_offset': c['timezone_offset'],
            'score_state': c['score_state'],
            'strain': c.get('score', {}).get('strain'),
            'kilojoule': c.get('score', {}).get('kilojoule'),
            'average_heart_rate': c.get('score', {}).get('average_heart_rate'),
            'max_heart_rate': c.get('score', {}).get('max_heart_rate'),
        }
        records.append(record)
    return pd.DataFrame(records)

df_cycles = flatten_cycles(raw_data['cycles'])
df_cycles['start'] = pd.to_datetime(df_cycles['start'])
df_cycles['end'] = pd.to_datetime(df_cycles['end'])
df_cycles['date'] = df_cycles['start'].dt.date

print(f"Cycles DataFrame: {df_cycles.shape}")
df_cycles.head()

Cycles DataFrame: (181, 10)


,cycle_id,start,end,timezone_offset,score_state,strain,kilojoule,average_heart_rate,max_heart_rate,date
0,1292957107,2026-02-04 07:31:08.530000+00:00,NaT,-05:00,SCORED,4.426687,6169.0913,57,128,2026-02-04
1,1290773527,2026-02-03 08:27:36.720000+00:00,2026-02-04 07:31:08.530000+00:00,-05:00,SCORED,6.539739,7269.8823,65,125,2026-02-03
2,1289096770,2026-02-02 08:28:33.720000+00:00,2026-02-03 08:27:36.720000+00:00,-05:00,SCORED,5.693672,7383.7417,61,132,2026-02-02
3,1287235600,2026-02-01 09:27:05.320000+00:00,2026-02-02 08:28:33.720000+00:00,-05:00,SCORED,6.907837,7461.3457,64,156,2026-02-01
4,1285433537,2026-01-31 08:14:06.510000+00:00,2026-02-01 09:27:05.320000+00:00,-05:00,SCORED,9.220865,8643.9550,66,169,2026-01-31


### 2.2 Recovery

In [5]:
# Examine a single recovery record
sample_recovery = raw_data['recovery'][0]
print("RECOVERY STRUCTURE:")
print(json.dumps(sample_recovery, indent=2))

RECOVERY STRUCTURE:
{
  "cycle_id": 1292957107,
  "sleep_id": "b048b364-3c6b-44c8-b718-9db233f399e1",
  "user_id": 13410036,
  "created_at": "2026-02-04T19:43:05.228Z",
  "updated_at": "2026-02-04T19:43:05.228Z",
  "score_state": "SCORED",
  "score": {
    "user_calibrating": false,
    "recovery_score": 93.0,
    "resting_heart_rate": 47.0,
    "hrv_rmssd_milli": 76.8064,
    "spo2_percentage": 96.8125,
    "skin_temp_celsius": 32.952667
  }
}


In [6]:
# Flatten recovery to DataFrame
def flatten_recovery(recovery_list):
    records = []
    for r in recovery_list:
        score = r.get('score', {})
        record = {
            'cycle_id': r['cycle_id'],
            'sleep_id': r['sleep_id'],
            'created_at': r['created_at'],
            'score_state': r['score_state'],
            'recovery_score': score.get('recovery_score'),
            'resting_heart_rate': score.get('resting_heart_rate'),
            'hrv_rmssd_milli': score.get('hrv_rmssd_milli'),
            'spo2_percentage': score.get('spo2_percentage'),
            'skin_temp_celsius': score.get('skin_temp_celsius'),
            'user_calibrating': score.get('user_calibrating'),
        }
        records.append(record)
    return pd.DataFrame(records)

df_recovery = flatten_recovery(raw_data['recovery'])
df_recovery['created_at'] = pd.to_datetime(df_recovery['created_at'])
df_recovery['date'] = df_recovery['created_at'].dt.date

print(f"Recovery DataFrame: {df_recovery.shape}")
df_recovery.head()

Recovery DataFrame: (179, 11)


,cycle_id,sleep_id,created_at,score_state,recovery_score,resting_heart_rate,hrv_rmssd_milli,spo2_percentage,skin_temp_celsius,user_calibrating,date
0,1292957107,b048b364-3c6b-44c8-b718-9db233f399e1,2026-02-04 19:43:05.228000+00:00,SCORED,93.0,47.0,76.806400,96.81250,32.952667,False,2026-02-04
1,1290773527,277c1675-8c45-4e57-8f2f-0d6233143c7c,2026-02-03 13:40:47.458000+00:00,SCORED,55.0,49.0,62.884315,96.40909,34.494167,False,2026-02-03
2,1289096770,30cbe93f-9be9-412c-86a6-fa59c5678a5a,2026-02-02 17:03:49.774000+00:00,SCORED,95.0,48.0,78.225110,97.53333,34.374000,False,2026-02-02
3,1287235600,494c6fac-45d2-4769-858b-3e0a629371a9,2026-02-01 18:17:56.175000+00:00,SCORED,95.0,50.0,77.895355,96.00000,34.010000,False,2026-02-01
4,1285433537,4eeb6a30-600d-463d-8a31-edeb15c72693,2026-01-31 19:26:21.670000+00:00,SCORED,70.0,55.0,66.981690,97.18000,33.734000,False,2026-01-31


### 2.3 Sleep

In [7]:
# Examine a single sleep record
sample_sleep = raw_data['sleep'][0]
print("SLEEP STRUCTURE:")
print(json.dumps(sample_sleep, indent=2))

SLEEP STRUCTURE:
{
  "id": "b048b364-3c6b-44c8-b718-9db233f399e1",
  "cycle_id": 1292957107,
  "v1_id": null,
  "user_id": 13410036,
  "created_at": "2026-02-04T19:43:05.228Z",
  "updated_at": "2026-02-04T19:43:05.228Z",
  "start": "2026-02-04T07:31:08.530Z",
  "end": "2026-02-04T18:24:49.640Z",
  "timezone_offset": "-05:00",
  "nap": false,
  "score_state": "SCORED",
  "score": {
    "stage_summary": {
      "total_in_bed_time_milli": 39221110,
      "total_awake_time_milli": 1980140,
      "total_no_data_time_milli": 0,
      "total_light_sleep_time_milli": 17554830,
      "total_slow_wave_sleep_time_milli": 7502420,
      "total_rem_sleep_time_milli": 12183720,
      "sleep_cycle_count": 9,
      "disturbance_count": 4
    },
    "sleep_needed": {
      "baseline_milli": 27873748,
      "need_from_sleep_debt_milli": 5443459,
      "need_from_recent_strain_milli": 293401,
      "need_from_recent_nap_milli": 0
    },
    "respiratory_rate": 14.135742,
    "sleep_performance_percentage

In [8]:
# Flatten sleep to DataFrame
def flatten_sleep(sleep_list):
    records = []
    for s in sleep_list:
        score = s.get('score', {})
        stage = score.get('stage_summary', {})
        need = score.get('sleep_needed', {})
        
        record = {
            'sleep_id': s['id'],
            'cycle_id': s['cycle_id'],
            'start': s['start'],
            'end': s['end'],
            'timezone_offset': s['timezone_offset'],
            'nap': s['nap'],
            'score_state': s['score_state'],
            # Stage summary (convert milli to hours)
            'total_in_bed_hours': stage.get('total_in_bed_time_milli', 0) / 3600000,
            'total_awake_hours': stage.get('total_awake_time_milli', 0) / 3600000,
            'total_light_sleep_hours': stage.get('total_light_sleep_time_milli', 0) / 3600000,
            'total_sws_hours': stage.get('total_slow_wave_sleep_time_milli', 0) / 3600000,
            'total_rem_hours': stage.get('total_rem_sleep_time_milli', 0) / 3600000,
            'sleep_cycles': stage.get('sleep_cycle_count'),
            'disturbances': stage.get('disturbance_count'),
            # Sleep metrics
            'respiratory_rate': score.get('respiratory_rate'),
            'sleep_performance_pct': score.get('sleep_performance_percentage'),
            'sleep_consistency_pct': score.get('sleep_consistency_percentage'),
            'sleep_efficiency_pct': score.get('sleep_efficiency_percentage'),
            # Sleep need
            'baseline_need_hours': need.get('baseline_milli', 0) / 3600000,
            'debt_need_hours': need.get('need_from_sleep_debt_milli', 0) / 3600000,
            'strain_need_hours': need.get('need_from_recent_strain_milli', 0) / 3600000,
        }
        records.append(record)
    return pd.DataFrame(records)

df_sleep = flatten_sleep(raw_data['sleep'])
df_sleep['start'] = pd.to_datetime(df_sleep['start'])
df_sleep['end'] = pd.to_datetime(df_sleep['end'])
df_sleep['date'] = df_sleep['start'].dt.date

print(f"Sleep DataFrame: {df_sleep.shape}")
print(f"\nNaps vs Main Sleep:")
print(df_sleep['nap'].value_counts())
df_sleep.head()

Sleep DataFrame: (195, 22)

Naps vs Main Sleep:
nap
False    179
True      16
Name: count, dtype: int64


,sleep_id,cycle_id,start,end,timezone_offset,nap,score_state,total_in_bed_hours,total_awake_hours,total_light_sleep_hours,...,sleep_cycles,disturbances,respiratory_rate,sleep_performance_pct,sleep_consistency_pct,sleep_efficiency_pct,baseline_need_hours,debt_need_hours,strain_need_hours,date
0,b048b364-3c6b-44c8-b718-9db233f399e1,1292957107,2026-02-04 07:31:08.530000+00:00,2026-02-04 18:24:49.640000+00:00,-05:00,False,SCORED,10.894753,0.550039,4.876342,...,9,4,14.135742,81.0,51.0,94.951340,7.742708,1.512072,0.081500,2026-02-04
1,277c1675-8c45-4e57-8f2f-0d6233143c7c,1290773527,2026-02-03 08:27:36.720000+00:00,2026-02-03 13:28:07.640000+00:00,-05:00,False,SCORED,5.008589,0.225278,2.327319,...,2,3,14.726562,69.0,50.0,95.502170,7.742790,0.000000,0.064665,2026-02-03
2,30cbe93f-9be9-412c-86a6-fa59c5678a5a,1289096770,2026-02-02 08:28:33.720000+00:00,2026-02-02 16:52:32.310000+00:00,-05:00,False,SCORED,8.399608,0.550575,3.989642,...,5,16,13.828125,88.0,73.0,93.445230,7.742872,0.000000,0.090060,2026-02-02
3,494c6fac-45d2-4769-858b-3e0a629371a9,1287235600,2026-02-01 09:27:05.320000+00:00,2026-02-01 18:09:59.010000+00:00,-05:00,False,SCORED,8.714914,0.458647,3.679106,...,2,12,14.238281,87.0,72.0,94.737210,7.742954,0.000000,0.166155,2026-02-01
4,4eeb6a30-600d-463d-8a31-edeb15c72693,1285433537,2026-01-31 08:14:06.510000+00:00,2026-01-31 19:02:08.600000+00:00,-05:00,False,SCORED,10.800581,0.717261,5.239989,...,8,9,15.410156,85.0,69.0,93.725426,7.743036,0.000000,0.164258,2026-01-31


### 2.4 Workouts

In [9]:
# Examine a single workout record
sample_workout = raw_data['workouts'][0]
print("WORKOUT STRUCTURE:")
print(json.dumps(sample_workout, indent=2))

WORKOUT STRUCTURE:
{
  "id": "920aea98-4a88-4ff4-a24f-0ad4a1430074",
  "v1_id": null,
  "user_id": 13410036,
  "created_at": "2026-02-01T23:05:04.872Z",
  "updated_at": "2026-02-01T23:05:20.022Z",
  "start": "2026-02-01T22:28:30.210Z",
  "end": "2026-02-01T22:44:59.230Z",
  "timezone_offset": "-05:00",
  "sport_name": "weightlifting",
  "score_state": "SCORED",
  "score": {
    "strain": 4.542832,
    "average_heart_rate": 101,
    "max_heart_rate": 133,
    "kilojoule": 213.70506,
    "percent_recorded": 0.9999798,
    "distance_meter": null,
    "altitude_gain_meter": null,
    "altitude_change_meter": null,
    "zone_durations": {
      "zone_zero_milli": 496000,
      "zone_one_milli": 493020,
      "zone_two_milli": 0,
      "zone_three_milli": 0,
      "zone_four_milli": 0,
      "zone_five_milli": 0
    }
  },
  "sport_id": 45
}


In [10]:
# Flatten workouts to DataFrame
def flatten_workouts(workout_list):
    records = []
    for w in workout_list:
        score = w.get('score', {})
        zones = score.get('zone_durations', {})
        
        record = {
            'workout_id': w['id'],
            'start': w['start'],
            'end': w['end'],
            'timezone_offset': w['timezone_offset'],
            'sport_name': w.get('sport_name'),
            'sport_id': w.get('sport_id'),
            'score_state': w['score_state'],
            # Workout metrics
            'strain': score.get('strain'),
            'average_heart_rate': score.get('average_heart_rate'),
            'max_heart_rate': score.get('max_heart_rate'),
            'kilojoule': score.get('kilojoule'),
            'distance_meter': score.get('distance_meter'),
            'altitude_gain_meter': score.get('altitude_gain_meter'),
            'percent_recorded': score.get('percent_recorded'),
            # HR Zones (convert milli to minutes)
            'zone_0_mins': zones.get('zone_zero_milli', 0) / 60000,
            'zone_1_mins': zones.get('zone_one_milli', 0) / 60000,
            'zone_2_mins': zones.get('zone_two_milli', 0) / 60000,
            'zone_3_mins': zones.get('zone_three_milli', 0) / 60000,
            'zone_4_mins': zones.get('zone_four_milli', 0) / 60000,
            'zone_5_mins': zones.get('zone_five_milli', 0) / 60000,
        }
        records.append(record)
    return pd.DataFrame(records)

df_workouts = flatten_workouts(raw_data['workouts'])
df_workouts['start'] = pd.to_datetime(df_workouts['start'])
df_workouts['end'] = pd.to_datetime(df_workouts['end'])
df_workouts['date'] = df_workouts['start'].dt.date
df_workouts['duration_mins'] = (df_workouts['end'] - df_workouts['start']).dt.total_seconds() / 60

print(f"Workouts DataFrame: {df_workouts.shape}")
print(f"\nWorkout Types:")
print(df_workouts['sport_name'].value_counts().head(10))
df_workouts.head()

Workouts DataFrame: (74, 22)

Workout Types:
sport_name
activity         43
walking          15
golf              5
weightlifting     3
soccer            2
skiing            1
basketball        1
commuting         1
dance             1
cycling           1
Name: count, dtype: int64


,workout_id,start,end,timezone_offset,sport_name,sport_id,score_state,strain,average_heart_rate,max_heart_rate,...,altitude_gain_meter,percent_recorded,zone_0_mins,zone_1_mins,zone_2_mins,zone_3_mins,zone_4_mins,zone_5_mins,date,duration_mins
0,920aea98-4a88-4ff4-a24f-0ad4a1430074,2026-02-01 22:28:30.210000+00:00,2026-02-01 22:44:59.230000+00:00,-05:00,weightlifting,45,SCORED,4.542832,101,133,...,None,0.999980,8.266667,8.217000,0.000000,0.000000,0.000000,0.0,2026-02-01,16.483667
1,c7855edf-d8c1-445a-8ca6-cd280b48202a,2026-01-27 21:13:30.550000+00:00,2026-01-27 22:05:00+00:00,-05:00,weightlifting,45,SCORED,9.640668,122,169,...,None,1.000000,7.400333,25.366667,10.483333,7.316833,0.933333,0.0,2026-01-27,51.490833
2,9e62d9cf-33af-4974-b8a4-3a79ee1a18d5,2026-01-27 00:46:30.510000+00:00,2026-01-27 01:01:29.520000+00:00,-05:00,activity,-1,SCORED,4.858775,111,135,...,None,0.999989,2.716667,12.266833,0.000000,0.000000,0.000000,0.0,2026-01-27,14.983500
3,3e281224-7e4a-4d3c-b8a4-e56d096753b2,2026-01-18 20:36:00.090000+00:00,2026-01-18 21:02:59.100000+00:00,-05:00,activity,-1,SCORED,5.311319,106,146,...,None,0.999994,10.483500,15.266667,1.233333,0.000000,0.000000,0.0,2026-01-18,26.983500
4,cb9a75fc-b3b2-486f-9966-95cf683968eb,2026-01-18 17:55:30.980000+00:00,2026-01-18 18:08:59.990000+00:00,-05:00,activity,-1,SCORED,4.887474,114,140,...,None,0.999988,2.683333,10.200167,0.600000,0.000000,0.000000,0.0,2026-01-18,13.483500


## 3. Data Quality Check

In [11]:
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

for name, df in [('Cycles', df_cycles), ('Recovery', df_recovery), 
                 ('Sleep', df_sleep), ('Workouts', df_workouts)]:
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Date Range: {df['date'].min()} to {df['date'].max()}")
    print(f"  Missing values:")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        for col, count in missing.items():
            print(f"    - {col}: {count} ({count/len(df)*100:.1f}%)")
    else:
        print("    None")

DATA QUALITY SUMMARY

Cycles:
  Shape: (181, 10)
  Date Range: 2025-08-08 to 2026-02-04
  Missing values:
    - end: 1 (0.6%)

Recovery:
  Shape: (179, 11)
  Date Range: 2025-08-09 to 2026-02-04
  Missing values:
    - skin_temp_celsius: 3 (1.7%)

Sleep:
  Shape: (195, 22)
  Date Range: 2025-08-09 to 2026-02-04
  Missing values:
    None

Workouts:
  Shape: (74, 22)
  Date Range: 2025-08-08 to 2026-02-01
  Missing values:
    - distance_meter: 74 (100.0%)
    - altitude_gain_meter: 74 (100.0%)


## 4. Relationship Analysis

In [12]:
# Check how tables relate via cycle_id
print("RELATIONSHIP ANALYSIS (via cycle_id)")
print("=" * 50)

cycles_ids = set(df_cycles['cycle_id'])
recovery_cycle_ids = set(df_recovery['cycle_id'])
sleep_cycle_ids = set(df_sleep['cycle_id'])

print(f"\nUnique cycle_ids in Cycles:   {len(cycles_ids)}")
print(f"Unique cycle_ids in Recovery: {len(recovery_cycle_ids)}")
print(f"Unique cycle_ids in Sleep:    {len(sleep_cycle_ids)}")

print(f"\nRecovery records matching Cycles: {len(recovery_cycle_ids & cycles_ids)}")
print(f"Sleep records matching Cycles:    {len(sleep_cycle_ids & cycles_ids)}")

RELATIONSHIP ANALYSIS (via cycle_id)

Unique cycle_ids in Cycles:   181
Unique cycle_ids in Recovery: 179
Unique cycle_ids in Sleep:    179

Recovery records matching Cycles: 179
Sleep records matching Cycles:    179


In [13]:
# Check sleep_id relationship
sleep_ids = set(df_sleep['sleep_id'])
recovery_sleep_ids = set(df_recovery['sleep_id'])

print("\nRELATIONSHIP via sleep_id:")
print(f"Unique sleep_ids in Sleep:    {len(sleep_ids)}")
print(f"Unique sleep_ids in Recovery: {len(recovery_sleep_ids)}")
print(f"Matching: {len(sleep_ids & recovery_sleep_ids)}")


RELATIONSHIP via sleep_id:
Unique sleep_ids in Sleep:    195
Unique sleep_ids in Recovery: 179
Matching: 179


## 5. Create Unified Daily DataFrame

In [14]:
# Create a unified daily view by joining on cycle_id
# Start with cycles as the base
df_daily = df_cycles[['cycle_id', 'date', 'strain', 'kilojoule', 
                       'average_heart_rate', 'max_heart_rate']].copy()
df_daily.columns = ['cycle_id', 'date', 'day_strain', 'day_kilojoule', 
                    'day_avg_hr', 'day_max_hr']

# Merge recovery
recovery_cols = ['cycle_id', 'recovery_score', 'resting_heart_rate', 
                 'hrv_rmssd_milli', 'spo2_percentage', 'skin_temp_celsius']
df_daily = df_daily.merge(df_recovery[recovery_cols], on='cycle_id', how='left')

# Merge main sleep (exclude naps)
main_sleep = df_sleep[df_sleep['nap'] == False].copy()
sleep_cols = ['cycle_id', 'total_in_bed_hours', 'total_sws_hours', 'total_rem_hours',
              'sleep_performance_pct', 'sleep_efficiency_pct', 'disturbances']
df_daily = df_daily.merge(main_sleep[sleep_cols], on='cycle_id', how='left')

# Sort by date
df_daily = df_daily.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Unified Daily DataFrame: {df_daily.shape}")
df_daily.head(10)

Unified Daily DataFrame: (181, 17)


,cycle_id,date,day_strain,day_kilojoule,day_avg_hr,day_max_hr,recovery_score,resting_heart_rate,hrv_rmssd_milli,spo2_percentage,skin_temp_celsius,total_in_bed_hours,total_sws_hours,total_rem_hours,sleep_performance_pct,sleep_efficiency_pct,disturbances
0,1292957107,2026-02-04,4.426687,6169.0913,57,128,93.0,47.0,76.806400,96.812500,32.952667,10.894753,2.084006,3.384367,81.0,94.951340,4.0
1,1290773527,2026-02-03,6.539739,7269.8823,65,125,55.0,49.0,62.884315,96.409090,34.494167,5.008589,1.866783,0.589208,69.0,95.502170,3.0
2,1289096770,2026-02-02,5.693672,7383.7417,61,132,95.0,48.0,78.225110,97.533330,34.374000,8.399608,2.075400,1.783992,88.0,93.445230,16.0
3,1287235600,2026-02-01,6.907837,7461.3457,64,156,95.0,50.0,77.895355,96.000000,34.010000,8.714914,2.234253,2.342908,87.0,94.737210,12.0
4,1285433537,2026-01-31,9.220865,8643.9550,66,169,70.0,55.0,66.981690,97.180000,33.734000,10.800581,1.858972,2.984358,85.0,93.725426,9.0
5,1283507040,2026-01-30,9.176349,8419.1010,65,153,69.0,49.0,66.490780,96.608696,33.634666,11.426958,2.284822,3.209083,82.0,92.704340,13.0
6,1281718255,2026-01-29,4.626066,6851.8457,63,112,67.0,50.0,63.918167,97.111115,33.758667,8.958264,2.650975,1.725669,80.0,94.411865,16.0
7,1279806879,2026-01-28,4.721075,7896.1772,62,134,57.0,53.0,59.206837,97.850000,33.296665,9.663286,2.125678,2.217886,87.0,95.418140,13.0
8,1277748235,2026-01-27,12.972899,9863.4100,67,169,88.0,47.0,78.645310,95.111115,33.409170,7.574592,1.892031,2.391808,80.0,95.250810,8.0
9,1276100606,2026-01-26,6.645034,7653.3022,58,144,94.0,47.0,84.857180,97.800000,32.969334,10.325567,1.775617,3.267969,84.0,94.996185,13.0


In [15]:
# Quick stats on the unified data
df_daily.describe()

,cycle_id,day_strain,day_kilojoule,day_avg_hr,day_max_hr,recovery_score,resting_heart_rate,hrv_rmssd_milli,spo2_percentage,skin_temp_celsius,total_in_bed_hours,total_sws_hours,total_rem_hours,sleep_performance_pct,sleep_efficiency_pct,disturbances
count,1.810000e+02,181.000000,181.000000,181.000000,181.000000,179.000000,179.000000,179.000000,179.000000,176.000000,179.000000,179.000000,179.000000,179.000000,179.000000,179.000000
mean,1.144778e+09,9.619295,8808.539054,67.580110,152.342541,61.877095,51.480447,68.319479,96.946782,33.484089,7.648764,1.959846,2.136451,71.743017,94.220973,9.592179
std,8.180449e+07,4.000043,2139.663802,6.709897,19.342349,22.865176,4.830080,11.732953,1.445553,0.528083,2.157554,0.413544,0.886484,15.542291,4.053390,4.328654
min,1.008933e+09,0.848157,3465.925000,54.000000,102.000000,1.000000,45.000000,26.426888,87.000000,32.191666,1.682320,0.784412,0.000000,7.000000,72.087800,0.000000
25%,1.074283e+09,6.566068,7557.703000,63.000000,139.000000,47.000000,48.000000,62.882177,96.379805,33.100000,6.305086,1.721594,1.639402,69.000000,93.430350,6.500000
50%,1.142572e+09,8.950556,8286.728000,66.000000,151.000000,66.000000,50.000000,70.441150,97.180000,33.569750,7.875449,1.933692,2.126208,77.000000,95.102190,9.000000
75%,1.213158e+09,12.315320,9640.076000,71.000000,164.000000,79.000000,53.500000,76.634745,97.866070,33.804500,9.161423,2.230824,2.740226,81.000000,96.589325,12.000000
max,1.292957e+09,20.217360,17722.525000,93.000000,201.000000,98.000000,69.000000,95.817580,99.250000,34.900000,13.867106,3.067675,4.417158,89.000000,100.000000,21.000000


## 6. Storage Recommendations

In [16]:
print("""
================================================================================
STORAGE RECOMMENDATIONS
================================================================================

Based on the data structure analysis, here are my recommendations:

OPTION 1: Parquet Files (RECOMMENDED)
--------------------------------------
Best for: Analysis, ML pipelines, efficient storage
Structure:
  data/processed/
    ├── cycles.parquet      (~50KB)
    ├── recovery.parquet    (~30KB) 
    ├── sleep.parquet       (~100KB)
    ├── workouts.parquet    (~50KB)
    └── daily_unified.parquet  (~50KB)

Pros:
  - Column-oriented (fast for analytics)
  - Compressed (5-10x smaller than CSV)
  - Preserves data types
  - Fast read/write with pandas
  
OPTION 2: SQLite Database
--------------------------------------
Best for: Complex queries, relationships, web apps
Structure:
  data/processed/whoop.db
    Tables: cycles, recovery, sleep, workouts
    Views: daily_unified

Pros:
  - SQL queries
  - Enforced relationships
  - Single file
  - Good for building apps on top

OPTION 3: CSV Files (Simple)
--------------------------------------
Best for: Excel users, simple sharing
Cons: Large files, no type preservation

================================================================================
MY RECOMMENDATION: Start with Parquet for analysis, add SQLite later if needed.
================================================================================
""")


STORAGE RECOMMENDATIONS

Based on the data structure analysis, here are my recommendations:

OPTION 1: Parquet Files (RECOMMENDED)
--------------------------------------
Best for: Analysis, ML pipelines, efficient storage
Structure:
  data/processed/
    ├── cycles.parquet      (~50KB)
    ├── recovery.parquet    (~30KB) 
    ├── sleep.parquet       (~100KB)
    ├── workouts.parquet    (~50KB)
    └── daily_unified.parquet  (~50KB)

Pros:
  - Column-oriented (fast for analytics)
  - Compressed (5-10x smaller than CSV)
  - Preserves data types
  - Fast read/write with pandas
  
OPTION 2: SQLite Database
--------------------------------------
Best for: Complex queries, relationships, web apps
Structure:
  data/processed/whoop.db
    Tables: cycles, recovery, sleep, workouts
    Views: daily_unified

Pros:
  - SQL queries
  - Enforced relationships
  - Single file
  - Good for building apps on top

OPTION 3: CSV Files (Simple)
--------------------------------------
Best for: Excel users,

## 7. Save Processed Data

In [17]:
# Create processed directory
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

# Save as Parquet
df_cycles.to_parquet(processed_dir / 'cycles.parquet', index=False)
df_recovery.to_parquet(processed_dir / 'recovery.parquet', index=False)
df_sleep.to_parquet(processed_dir / 'sleep.parquet', index=False)
df_workouts.to_parquet(processed_dir / 'workouts.parquet', index=False)
df_daily.to_parquet(processed_dir / 'daily_unified.parquet', index=False)

print("Saved Parquet files:")
for f in processed_dir.glob('*.parquet'):
    print(f"  {f.name}: {f.stat().st_size / 1024:.1f} KB")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
# Also save as CSV for easy inspection
df_daily.to_csv(processed_dir / 'daily_unified.csv', index=False)
print(f"\nAlso saved: daily_unified.csv")

## 8. Quick Visualization Preview

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Recovery over time
ax1 = axes[0, 0]
ax1.plot(df_daily['date'], df_daily['recovery_score'], alpha=0.7)
ax1.set_title('Recovery Score Over Time')
ax1.set_ylabel('Recovery %')
ax1.tick_params(axis='x', rotation=45)

# Strain over time
ax2 = axes[0, 1]
ax2.plot(df_daily['date'], df_daily['day_strain'], alpha=0.7, color='orange')
ax2.set_title('Daily Strain Over Time')
ax2.set_ylabel('Strain')
ax2.tick_params(axis='x', rotation=45)

# HRV distribution
ax3 = axes[1, 0]
ax3.hist(df_daily['hrv_rmssd_milli'].dropna(), bins=30, alpha=0.7, color='green')
ax3.set_title('HRV Distribution')
ax3.set_xlabel('HRV (ms)')

# Recovery vs Strain scatter
ax4 = axes[1, 1]
ax4.scatter(df_daily['recovery_score'], df_daily['day_strain'], alpha=0.5)
ax4.set_title('Recovery vs Strain')
ax4.set_xlabel('Recovery %')
ax4.set_ylabel('Strain')

plt.tight_layout()
plt.savefig(processed_dir / 'quick_overview.png', dpi=100)
plt.show()

print("\nSaved: quick_overview.png")

## Next Steps

Now that you have clean, structured data, you can:

1. **Build a Recovery Prediction Model** - Use `daily_unified.parquet` to predict recovery based on sleep, strain, and other factors
2. **Time Series Analysis** - Look for patterns, seasonality, trends
3. **Feature Engineering** - Add rolling averages, lag features, etc.
4. **Dashboard** - Build a Streamlit or Plotly dashboard